# Caso sintético de arsénico

Este notebook reproduce el mismo caso sintético utilizado en las clases de 2026. Todos los valores y coordenadas son ficticios y no representan pozos reales.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 20260819
rng = np.random.default_rng(SEED)
N = 1000
x = rng.uniform(0, 20, N)
y = rng.uniform(0, 15, N)
regional = 8 + 0.45*x + 0.20*y
anomalia_1 = 45*np.exp(-(((x-6)/3.2)**2 + ((y-10)/2.8)**2))
anomalia_2 = 25*np.exp(-(((x-15)/2.8)**2 + ((y-4)/2.5)**2))
ruido = rng.lognormal(mean=2.05, sigma=0.55, size=N)
ruido = ruido - ruido.mean()
As = np.clip(regional + anomalia_1 + anomalia_2 + ruido, 1, None)

poblacion = pd.DataFrame({
    'pozo': [f'P{i:04d}' for i in range(1, N+1)],
    'x_km': x, 'y_km': y, 'as_ug_l': As
})
campania = poblacion.sample(frac=1, random_state=2026).reset_index(drop=True).iloc[:50].copy()
campania.head()

In [ ]:
v = campania['as_ug_l'].to_numpy()
pd.Series({
    'n': len(v),
    'media': np.mean(v),
    'mediana': np.median(v),
    'SD': np.std(v, ddof=1),
    'asimetria': stats.skew(v, bias=False),
    'exceso_curtosis': stats.kurtosis(v, fisher=True, bias=False),
}).round(3)

Los valores de control deben ser aproximadamente: media 18.013, mediana 15.736, SD 10.965, asimetría 1.662 y exceso de curtosis 3.130. Si no aparecen, el caso dejó de estar sincronizado con las clases.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].hist(v, bins='fd', color='#446681', edgecolor='white')
axes[0].set(xlabel='As sintético (µg/L)', ylabel='Frecuencia', title='Campaña de 50 pozos')
m = axes[1].scatter(campania['x_km'], campania['y_km'], c=campania['as_ug_l'], cmap='viridis', s=35)
axes[1].set(xlabel='x (km)', ylabel='y (km)', title='Cobertura espacial', aspect='equal')
fig.colorbar(m, ax=axes[1], label='As sintético (µg/L)')
plt.show()

## Para continuar

1. Compare histograma, ECDF, KDE y boxplot.
2. Ajuste una normal y compare probabilidades del modelo con proporciones observadas.
3. Repita el muestreo muchas veces y grafique la distribución de las medias.
4. Permute los valores entre coordenadas y compruebe qué resúmenes marginales se conservan.